# Problema 2 – Regresión: Estimación de Edad a partir de Imágenes Faciales

## Objetivo
Desarrollar un modelo de regresión basado en redes neuronales convolucionales (CNN) para estimar la edad de una persona a partir de imágenes faciales.

## Estructura del notebook
1. Análisis preliminar del problema
2. Análisis exploratorio de datos (EDA)
3. División del dataset en train / val / test
4. Preprocesamiento con `Dataset` y `DataLoader`
5. Construcción y entrenamiento del modelo CNN
6. Evaluación con MAE, RMSE y R²
7. Prueba con muestra individual
8. Conclusiones

In [ ]:
# En Lightning.ai no se necesita google.colab
# Sube UTKFace.zip al panel de archivos de la izquierda
import zipfile, os

zip_name = "UTKFace.zip"  # Ajusta el nombre si es diferente
if not os.path.exists(zip_name):
    raise FileNotFoundError(
        f"No se encontró {zip_name}. "
        "Sube el archivo al panel de archivos de Lightning.ai (panel izquierdo)."
    )
uploaded = {zip_name: None}  # Compatibilidad con el resto del código
print(f"Archivo encontrado: {zip_name}")

In [ ]:
import os
import re
import csv
import math
import shutil
import random
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import r2_score

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

## 1. Análisis preliminar del problema

### a. Justificación
Este problema corresponde a una tarea de **regresión supervisada**, ya que el objetivo es predecir una variable numérica continua: la **edad** de la persona en la imagen. A diferencia de la clasificación, donde la salida es una categoría discreta, aquí se requiere estimar un valor numérico en años.

La **variable objetivo (target)** es la **edad** asociada a cada imagen facial.

### b. Características de entrada
Las entradas del modelo son imágenes faciales en color. Estas imágenes contienen información útil para la estimación de edad, como textura de la piel, rasgos faciales, arrugas, proporciones y otros patrones asociados al envejecimiento.

### c. Protocolo de adquisición o generación de los datos
Para este trabajo se utiliza un dataset de imágenes faciales donde la edad exacta está codificada en el nombre del archivo. Esto permite extraer directamente la etiqueta numérica y formular el problema como regresión.

In [ ]:
zip_name = list(uploaded.keys())[0]

raw_dir = Path("raw_data")
if raw_dir.exists():
    shutil.rmtree(raw_dir)
raw_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(raw_dir)

print("ZIP descomprimido en:", raw_dir)

In [ ]:
all_files = []
for root, dirs, files_ in os.walk(raw_dir):
    for f in files_:
        all_files.append(Path(root) / f)

print("Total de archivos encontrados:", len(all_files))
print("Primeros archivos:")
for p in all_files[:10]:
    print(p)

In [ ]:
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}

all_images = [p for p in all_files if p.suffix.lower() in EXTENSIONS]

def parse_age(filename: str):
    match = re.match(r"^(\d+)_", filename)
    if match:
        age = int(match.group(1))
        if 0 <= age <= 116:
            return age
    return None

valid_images = [p for p in all_images if parse_age(p.name) is not None]

print("Imágenes encontradas:", len(all_images))
print("Imágenes válidas para regresión:", len(valid_images))

if len(valid_images) == 0:
    raise ValueError(
        "No se encontraron imágenes con formato válido. "
        "Los nombres deben comenzar con la edad, por ejemplo: 25_0_2_xxx.jpg"
    )

## 2. División del dataset

Se utiliza una división **70% entrenamiento, 15% validación y 15% prueba**.  
La división se realiza a partir de las rutas de archivo, sin cargar las imágenes completas en memoria, lo cual hace el proceso más eficiente.

In [ ]:
OUTPUT_DIR = Path("dataset")

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

COPY_FILES = True
CHUNK_SIZE = 500

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
random.shuffle(valid_images)

total = len(valid_images)
n_train = int(total * TRAIN_RATIO)
n_val   = int(total * VAL_RATIO)
n_test  = total - n_train - n_val

splits = {
    "train": valid_images[:n_train],
    "val": valid_images[n_train:n_train + n_val],
    "test": valid_images[n_train + n_val:]
}

print("División calculada:")
for name, files_ in splits.items():
    pct = len(files_) / total * 100
    print(f"{name:>5}: {len(files_):>6} imágenes ({pct:.1f}%)")

In [ ]:
for split_name in splits:
    dest = OUTPUT_DIR / split_name
    dest.mkdir(parents=True, exist_ok=True)

def process_in_chunks(file_list, dest_folder, operation="copy", chunk_size=500):
    total_files = len(file_list)
    errors = []
    fn = shutil.copy2 if operation == "copy" else shutil.move

    with tqdm(total=total_files, unit="img", dynamic_ncols=True) as pbar:
        for start in range(0, total_files, chunk_size):
            chunk = file_list[start:start + chunk_size]
            for src in chunk:
                dst = dest_folder / src.name
                if dst.exists():
                    dst = dest_folder / f"{src.stem}_{src.stat().st_ino}{src.suffix}"
                try:
                    fn(src, dst)
                except Exception as e:
                    errors.append((src, str(e)))
                pbar.update(1)
    return errors

operation = "copy" if COPY_FILES else "move"
all_errors = {}

for split_name, file_list in splits.items():
    print(f"\nProcesando [{split_name}] — {len(file_list):,} archivos")
    dest_folder = OUTPUT_DIR / split_name
    errs = process_in_chunks(file_list, dest_folder, operation, CHUNK_SIZE)
    if errs:
        all_errors[split_name] = errs

print("\nProceso completado.")

In [ ]:
print("Verificación final:")
grand_total = 0
for split_name in splits:
    dest = OUTPUT_DIR / split_name
    count = sum(1 for f in dest.iterdir() if f.suffix.lower() in EXTENSIONS)
    expected = len(splits[split_name])
    status = "OK" if count == expected else "ERROR"
    print(f"{split_name:>5}: {count} / {expected} -> {status}")
    grand_total += count

print("Total final:", grand_total)

log_path = OUTPUT_DIR / "split_log.csv"
with open(log_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["filename", "split"])
    for split_name, file_list in splits.items():
        for p in file_list:
            writer.writerow([p.name, split_name])

print("Log guardado en:", log_path)

if all_errors:
    print("Errores detectados:", all_errors)
else:
    print("Sin errores. Dataset listo para entrenar.")

## 3. Análisis exploratorio de datos (EDA)

En esta sección se estudia la distribución de edades, se visualizan ejemplos del dataset y se analiza la variabilidad de las imágenes.

In [ ]:
rows = []
for split_name in ["train", "val", "test"]:
    folder = OUTPUT_DIR / split_name
    for p in folder.iterdir():
        if p.suffix.lower() in EXTENSIONS:
            age = parse_age(p.name)
            if age is not None:
                rows.append({"split": split_name, "image_path": str(p), "age": age})

df = pd.DataFrame(rows)
df.head()

In [ ]:
print(df["age"].describe())

plt.figure(figsize=(10, 5))
plt.hist(df["age"], bins=40, edgecolor="black")
plt.title("Distribución de edades")
plt.xlabel("Edad")
plt.ylabel("Frecuencia")
plt.show()

### Interpretación
La distribución de edades permite identificar posibles desbalances. Si ciertos rangos de edad están sobrerrepresentados, el modelo tenderá a aprender mejor esos casos y podría fallar más en edades menos frecuentes.

In [ ]:
sample_paths = df.sample(min(9, len(df)), random_state=SEED).reset_index(drop=True)

plt.figure(figsize=(12, 12))
for i in range(len(sample_paths)):
    img = Image.open(sample_paths.loc[i, "image_path"]).convert("RGB")
    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.title(f"Edad: {sample_paths.loc[i, 'age']}")
    plt.axis("off")
plt.tight_layout()
plt.show()

### Interpretación
Las imágenes presentan variabilidad en iluminación, expresión, pose, resolución y calidad visual. Esta heterogeneidad aumenta la complejidad del problema, pero hace el escenario más cercano a una aplicación real.

In [ ]:
sample_dims = []
for p in df["image_path"].sample(min(200, len(df)), random_state=SEED):
    img = Image.open(p).convert("RGB")
    sample_dims.append(img.size)

widths = [w for w, h in sample_dims]
heights = [h for w, h in sample_dims]

print("Ancho promedio:", np.mean(widths))
print("Alto promedio:", np.mean(heights))
print("Canales esperados: 3 (RGB)")

## 4. Procesamiento de datos

Las imágenes se redimensionan a un tamaño fijo y se normalizan usando los valores estándar de ImageNet. En entrenamiento se aplica data augmentation moderado para mejorar la capacidad de generalización.

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32
NUM_WORKERS = 2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("Transformaciones definidas.")

In [ ]:
class AgeDataset(Dataset):
    """
    Dataset para imágenes con etiqueta de edad en el nombre del archivo.
    Formato esperado: [age]_[gender]_[race]_[datetime].jpg
    """

    def __init__(self, root_dir: Path, transform=None):
        self.transform = transform
        self.samples = []

        for img_path in Path(root_dir).iterdir():
            if img_path.suffix.lower() not in EXTENSIONS:
                continue
            age = self._parse_age(img_path.name)
            if age is not None:
                self.samples.append((img_path, float(age)))

        print(f"[{root_dir.name}] {len(self.samples)} imágenes cargadas")

    @staticmethod
    def _parse_age(filename: str):
        match = re.match(r'^(\d+)_', filename)
        if match:
            age = int(match.group(1))
            if 0 <= age <= 116:
                return age
        return None

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, age = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(age, dtype=torch.float32)

In [ ]:
train_dataset = AgeDataset(OUTPUT_DIR / "train", transform=train_transform)
val_dataset   = AgeDataset(OUTPUT_DIR / "val", transform=val_transform)
test_dataset  = AgeDataset(OUTPUT_DIR / "test", transform=val_transform)

In [ ]:
def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)

sample_img, sample_age = train_dataset[0]
print("Shape:", sample_img.shape)
print("Edad:", sample_age.item())

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
indices = np.random.choice(len(train_dataset), 10, replace=False)

for ax, idx in zip(axes.flat, indices):
    img_t, age = train_dataset[idx]
    img_vis = denormalize(img_t).permute(1, 2, 0).numpy()
    ax.imshow(img_vis)
    ax.set_title(f"Edad: {int(age.item())}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda",
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda",
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda",
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
batch_imgs, batch_ages = next(iter(train_loader))
print("batch_imgs.shape:", batch_imgs.shape)
print("batch_ages.shape:", batch_ages.shape)
print("Primeras edades:", batch_ages[:8].int().tolist())

## 5. Modelo CNN para regresión

Se define una CNN base con tres bloques convolucionales y una cabeza densa para regresión. La salida tiene una sola neurona con activación lineal, ya que el objetivo es predecir una edad numérica.

In [ ]:
class AgeRegressionCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * (IMG_SIZE // 8) * (IMG_SIZE // 8), 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.regressor(x)
        return x

model = AgeRegressionCNN().to(DEVICE)
model

### Justificación de la arquitectura
- Las capas convolucionales extraen patrones faciales locales.
- `MaxPooling` reduce dimensionalidad y conserva información relevante.
- `Batch Normalization` ayuda a estabilizar el entrenamiento.
- `Dropout` reduce el riesgo de overfitting.
- La salida tiene una sola neurona porque se predice un valor escalar.

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
NUM_EPOCHS = 10
best_model_path = "best_age_model.pth"

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    running_mae = 0.0
    total = 0

    for batch_imgs, batch_ages in loader:
        batch_imgs = batch_imgs.to(device)
        batch_ages = batch_ages.to(device)

        optimizer.zero_grad()
        preds = model(batch_imgs).squeeze(1)

        loss = criterion(preds, batch_ages)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch_imgs.size(0)
        running_mae += torch.abs(preds - batch_ages).sum().item()
        total += batch_imgs.size(0)

    return running_loss / total, running_mae / total


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_mae = 0.0
    total = 0

    preds_all = []
    targets_all = []

    with torch.no_grad():
        for batch_imgs, batch_ages in loader:
            batch_imgs = batch_imgs.to(device)
            batch_ages = batch_ages.to(device)

            preds = model(batch_imgs).squeeze(1)
            loss = criterion(preds, batch_ages)

            running_loss += loss.item() * batch_imgs.size(0)
            running_mae += torch.abs(preds - batch_ages).sum().item()
            total += batch_imgs.size(0)

            preds_all.extend(preds.cpu().numpy().tolist())
            targets_all.extend(batch_ages.cpu().numpy().tolist())

    rmse = math.sqrt(np.mean((np.array(preds_all) - np.array(targets_all)) ** 2))
    r2 = r2_score(targets_all, preds_all)

    return running_loss / total, running_mae / total, rmse, r2, preds_all, targets_all

In [ ]:
history = {
    "train_loss": [],
    "val_loss": [],
    "train_mae": [],
    "val_mae": []
}

best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):
    train_loss, train_mae = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_mae, val_rmse, val_r2, _, _ = evaluate(model, val_loader, criterion, DEVICE)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_mae"].append(train_mae)
    history["val_mae"].append(val_mae)

    print(f"Época {epoch+1}/{NUM_EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f} | Train MAE: {train_mae:.2f}")
    print(f"  Val   Loss: {val_loss:.4f} | Val   MAE: {val_mae:.2f} | Val RMSE: {val_rmse:.2f} | Val R²: {val_r2:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print("  Mejor modelo guardado.")

In [ ]:
epochs_range = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_range, history["train_loss"], label="Train Loss", marker="o")
ax1.plot(epochs_range, history["val_loss"], label="Val Loss", marker="o")
ax1.set_title("MSE Loss por época")
ax1.set_xlabel("Época")
ax1.set_ylabel("MSE")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history["train_mae"], label="Train MAE", marker="s")
ax2.plot(epochs_range, history["val_mae"], label="Val MAE", marker="s")
ax2.set_title("MAE por época")
ax2.set_xlabel("Época")
ax2.set_ylabel("MAE (años)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

gap = history["val_mae"][-1] - history["train_mae"][-1]
print("Diagnóstico final:")
print(f"Train MAE: {history['train_mae'][-1]:.2f}")
print(f"Val   MAE: {history['val_mae'][-1]:.2f}")
print(f"Gap      : {gap:.2f}")

if gap > 3:
    print("Posible overfitting.")
elif history["train_mae"][-1] > 8:
    print("Posible underfitting.")
else:
    print("Buen balance entre ajuste y generalización.")

### Interpretación de las curvas
Si la pérdida de entrenamiento disminuye mucho más que la de validación, hay evidencia de sobreajuste. Si ambas pérdidas permanecen altas, el modelo presenta subajuste. El comportamiento ideal es que ambas curvas disminuyan y se mantengan relativamente cercanas.

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()

test_loss, test_mae, test_rmse, test_r2, all_preds, all_targets = evaluate(model, test_loader, criterion, DEVICE)

metrics_df = pd.DataFrame({
    "Métrica": ["MAE", "RMSE", "R²"],
    "Valor": [test_mae, test_rmse, test_r2]
})

metrics_df

## 6. Evaluación final del modelo

Las métricas usadas son:
- **MAE**: error absoluto medio en años.
- **RMSE**: raíz del error cuadrático medio.
- **R²**: proporción de variabilidad explicada por el modelo.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(all_targets, all_preds, alpha=0.3)
min_age = min(all_targets)
max_age = max(all_targets)
plt.plot([min_age, max_age], [min_age, max_age], linestyle="--")
plt.xlabel("Edad real")
plt.ylabel("Edad predicha")
plt.title("Predicción vs valor real")
plt.show()

### Interpretación
Un buen modelo debe producir puntos cercanos a la diagonal. Mientras más dispersión haya, mayor será el error de predicción.

In [ ]:
sample_test_path = df[df["split"] == "test"].sample(1, random_state=SEED).iloc[0]["image_path"]
sample_real_age = parse_age(Path(sample_test_path).name)

image_pil = Image.open(sample_test_path).convert("RGB")
image_tensor = val_transform(image_pil).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    pred_age = model(image_tensor).item()

plt.figure(figsize=(5, 5))
plt.imshow(image_pil)
plt.axis("off")
plt.title(f"Edad real: {sample_real_age} | Edad predicha: {pred_age:.2f}")
plt.show()

In [ ]:
def predict_pil_image(img_pil):
    x = val_transform(img_pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        return model(x).item()

img_np = np.array(image_pil)

bright = np.clip(img_np * 1.25, 0, 255).astype(np.uint8)
small = np.array(Image.fromarray(img_np).resize((80, 80)).resize(image_pil.size))
rotated = np.array(image_pil.rotate(10))

variants = {
    "Original": Image.fromarray(img_np),
    "Más iluminación": Image.fromarray(bright),
    "Menor escala": Image.fromarray(small),
    "Rotada": Image.fromarray(rotated)
}

plt.figure(figsize=(12, 8))
for i, (name, img_var) in enumerate(variants.items(), 1):
    pred = predict_pil_image(img_var)
    plt.subplot(2, 2, i)
    plt.imshow(img_var)
    plt.title(f"{name}\nPredicción: {pred:.2f}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 7. Prueba con muestra artificial

La predicción sobre una imagen individual permite verificar si el modelo produce una edad razonable. Al modificar iluminación, escala u orientación, la predicción puede variar porque estos cambios alteran los patrones visuales usados por la red.

Si el modelo es robusto, las variaciones no deberían causar cambios extremos en la edad estimada.

## 8. Conclusiones

1. La estimación de edad a partir de imágenes faciales corresponde a un problema de regresión.
2. La distribución del target influye directamente en la capacidad del modelo para generalizar.
3. El uso de `Dataset` y `DataLoader` permite construir un pipeline eficiente y reproducible.
4. Las métricas MAE, RMSE y R² permiten evaluar el modelo desde perspectivas complementarias.
5. Las variaciones de iluminación, orientación y escala afectan la predicción, por lo que el preprocesamiento y la regularización son fundamentales.